# Real-Time Fraud Detection: Feature Engineering

## 1. Principles of Fraud Feature Engineering
Effective fraud detection features must meet two criteria:
1. **High Discriminative Power**: Differentiate anomalous / adversarial transaction patterns from regular consumer activity.
2. **Sub-Millisecond Inference Efficiency**: Be computable in real-time during payment gateway authorization (<10ms SLA).

In this notebook, we evaluate:
- Amount transformations (log scaling, zero/micro authorizations, round amounts)
- Temporal transformations (cyclical sine/cosine hour encoding, night-time flags)
- Latent vector interaction dynamics (norm, product, and contrast signals)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_credit_card_data
from src.data.split import time_based_split
from src.features.engineer import TransactionFeatureEngineer

# 1. Load data
df = load_credit_card_data(data_path='../data/raw/creditcard.csv')

# 2. Strict Out-of-Time split to prevent leakage
train_df, val_df, test_df = time_based_split(df, val_size=0.15, test_size=0.15)

# 3. Fit feature engineer ONLY on train
engineer = TransactionFeatureEngineer()
X_train_raw = train_df.drop(columns=['Class'])
y_train = train_df['Class']

engineer.fit(X_train_raw)
X_train = engineer.transform(X_train_raw)
X_val = engineer.transform(val_df.drop(columns=['Class']))
X_test = engineer.transform(test_df.drop(columns=['Class']))

print(f"Raw feature count: {X_train_raw.shape[1]}")
print(f"Engineered feature count: {X_train.shape[1]}")

## 2. Inspecting Engineered Signals
Visualizing the discriminative power of engineered features between legitimate and fraudulent transactions.

In [ ]:
# Combine engineered features with label for exploratory visualization
train_engineered = X_train.copy()
train_engineered['Class'] = y_train.values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Latent vector norm
sns.kdeplot(data=train_engineered, x='v_vector_norm', hue='Class', common_norm=False, ax=axes[0, 0], palette=['#2b5c8f', '#d9534f'])
axes[0, 0].set_title('Latent Vector Norm (Euclidean Distance from Origin)', fontweight='bold')

# 2. Amount to median ratio (log scale)
train_engineered['log_amount_ratio'] = np.log1p(train_engineered['amount_to_median_ratio'])
sns.boxplot(data=train_engineered, x='Class', y='log_amount_ratio', ax=axes[0, 1], palette=['#2b5c8f', '#d9534f'], showfliers=False)
axes[0, 1].set_title('Amount to Median Ratio [log1p]', fontweight='bold')

# 3. Nocturnal Fraud Distribution
night_ct = train_engineered.groupby('is_night')['Class'].mean() * 100
sns.barplot(x=['Daytime (06:00-24:00)', 'Night-time (00:00-06:00)'], y=night_ct.values, ax=axes[1, 0], palette=['#2b5c8f', '#d9534f'])
axes[1, 0].set_title('Fraud Rate: Night-time vs Daytime (%)', fontweight='bold')
axes[1, 0].set_ylabel('Fraud Rate (%)')

# 4. High-leverage contrast feature (V4 - V14)
sns.kdeplot(data=train_engineered, x='v4_v14_diff', hue='Class', common_norm=False, ax=axes[1, 1], palette=['#2b5c8f', '#d9534f'])
axes[1, 1].set_title('Contrast Feature (V4 - V14 Separability)', fontweight='bold')

plt.tight_layout()
plt.show()